# Cold-Start Analysis

This notebook loads cold-start result sessions for RPG and SASRec from both the shared group artifacts tree and the repo-local artifacts tree, then builds reporting tables and plots for the latest available run of each dataset, model, and config track.

Expected layouts:

```text
<artifact-root>/<track>/<dataset>/<session>/tables/cold_start_summary.json
<artifact-root>/<session>/tables/cold_start_summary.json
```

The second form is the older single-session layout and is labeled `legacy` in the tables below.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd().resolve()
while ROOT.name not in {"RPG", "RPG-uva"} and ROOT.parent != ROOT:
    ROOT = ROOT.parent

ARTIFACT_SOURCES = [
    {
        "model": "RPG",
        "root": Path("/projects/prjs2120/groups/group_16/artifacts/rpg/cold_start"),
    },
    {
        "model": "RPG",
        "root": ROOT / "artifacts" / "rpg" / "cold_start",
    },
    {
        "model": "SASRec",
        "root": Path("/projects/prjs2120/groups/group_16/artifacts/sasrec/cold_start"),
    },
    {
        "model": "SASRec",
        "root": ROOT / "artifacts" / "sasrec" / "cold_start",
    },
]

seen_sources = set()
ARTIFACT_SOURCES = [
    source
    for source in ARTIFACT_SOURCES
    if (source["model"], source["root"]) not in seen_sources
    and not seen_sources.add((source["model"], source["root"]))
]

METRIC_ORDER = ["recall@5", "ndcg@5", "recall@10", "ndcg@10"]
PRIMARY_METRIC = "ndcg@10"
TRACK_LABELS = {
    "released_readme": "released README",
    "paper_appendix": "paper appendix",
    "legacy": "legacy",
}
DATASET_LABELS = {
    "Sports_and_Outdoors": "Sports",
    "sports_and_outdoors": "Sports",
    "Beauty": "Beauty",
    "beauty": "Beauty",
    "Toys_and_Games": "Toys",
    "toys_and_games": "Toys",
    "CDs_and_Vinyl": "CDs",
    "cds_and_vinyl": "CDs",
}
DATASET_ORDER = ["Sports", "Beauty", "Toys", "CDs"]
MODEL_ORDER = ["RPG", "SASRec"]
TRACK_ORDER = ["released README", "paper appendix", "legacy"]


In [ ]:
def _dataset_label(raw_value: str | None) -> str:
    if raw_value is None:
        return "Unknown"
    return DATASET_LABELS.get(raw_value, raw_value)


def _track_label(raw_value: str) -> str:
    return TRACK_LABELS.get(raw_value, raw_value)


def _model_label(payload: dict, source_model: str) -> str:
    raw_model = str(payload.get("model") or source_model)
    if raw_model.lower().startswith("sasrec"):
        return "SASRec"
    if raw_model.upper() == "RPG":
        return "RPG"
    return raw_model


def _ordered(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    if frame.empty:
        return frame
    sort_frame = frame.copy()
    sort_frame["_model_rank"] = sort_frame["model"].map({name: idx for idx, name in enumerate(MODEL_ORDER)}).fillna(len(MODEL_ORDER))
    sort_frame["_dataset_rank"] = sort_frame["dataset"].map({name: idx for idx, name in enumerate(DATASET_ORDER)}).fillna(len(DATASET_ORDER))
    sort_frame["_track_rank"] = sort_frame["track_label"].map({name: idx for idx, name in enumerate(TRACK_ORDER)}).fillna(len(TRACK_ORDER))
    sort_columns = ["_dataset_rank", "_model_rank", "_track_rank"] + columns
    return sort_frame.sort_values(sort_columns).drop(columns=["_model_rank", "_dataset_rank", "_track_rank"]).reset_index(drop=True)


def _parse_summary(summary_path: Path, artifact_root: Path, source_model: str) -> tuple[dict, list[dict]]:
    payload = json.loads(summary_path.read_text())
    session_root = summary_path.parents[1]
    relative_parts = summary_path.relative_to(artifact_root).parts

    if len(relative_parts) >= 5 and relative_parts[-2] == "tables":
        track = relative_parts[0]
        dataset_slug = relative_parts[1]
        session = relative_parts[2]
    else:
        track = "legacy"
        dataset_slug = payload.get("category") or payload.get("dataset") or "unknown"
        session = session_root.name

    model = _model_label(payload, source_model)
    dataset = _dataset_label(payload.get("category") or dataset_slug)
    overall_results = payload.get("overall_results", {})
    group_rows = payload.get("group_rows", [])
    checkpoint_path = payload.get("checkpoint_path")
    row = {
        "model": model,
        "artifact_root": str(artifact_root),
        "summary_path": str(summary_path),
        "session_root": str(session_root),
        "session": session,
        "track": track,
        "track_label": _track_label(track),
        "dataset_slug": dataset_slug,
        "dataset": dataset,
        "category": payload.get("category"),
        "bucket_spec": payload.get("bucket_spec"),
        "checkpoint_path": checkpoint_path,
        "checkpoint_name": None if checkpoint_path is None else Path(checkpoint_path).name,
        "plot_metric": payload.get("plot_metric"),
    }
    for metric_name in METRIC_ORDER:
        row[metric_name] = overall_results.get(metric_name)
    row["n_buckets"] = len(group_rows)

    bucket_rows = []
    for bucket in group_rows:
        bucket_row = {
            "model": model,
            "artifact_root": str(artifact_root),
            "summary_path": str(summary_path),
            "session_root": str(session_root),
            "session": session,
            "track": track,
            "track_label": _track_label(track),
            "dataset_slug": dataset_slug,
            "dataset": dataset,
            "bucket_label": bucket.get("bucket_label"),
            "bucket_range": bucket.get("bucket_range"),
            "bucket_min_count": bucket.get("bucket_min_count"),
            "bucket_max_count": bucket.get("bucket_max_count"),
            "n_examples": bucket.get("n_examples", 0),
        }
        for metric_name in METRIC_ORDER + ["n_visited_items"]:
            bucket_row[metric_name] = bucket.get(metric_name)
        bucket_rows.append(bucket_row)
    return row, bucket_rows


run_rows = []
bucket_rows = []
for source in ARTIFACT_SOURCES:
    artifact_root = source["root"]
    if not artifact_root.exists():
        continue
    for summary_path in sorted(artifact_root.rglob("cold_start_summary.json")):
        parsed_run, parsed_buckets = _parse_summary(summary_path, artifact_root, source["model"])
        run_rows.append(parsed_run)
        bucket_rows.extend(parsed_buckets)

all_runs = pd.DataFrame(run_rows)
all_buckets = pd.DataFrame(bucket_rows)

if all_runs.empty:
    joined_roots = "\n".join(f"{source['model']}: {source['root']}" for source in ARTIFACT_SOURCES)
    print(f"No cold_start_summary.json files found under:\n{joined_roots}")
    latest_runs = pd.DataFrame()
    latest_buckets = pd.DataFrame()
else:
    latest_runs = (
        all_runs.sort_values(["model", "dataset", "track", "session"])
        .groupby(["model", "dataset", "track"], as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )
    latest_keys = latest_runs[["model", "dataset", "track", "session"]]
    latest_buckets = all_buckets.merge(latest_keys, on=["model", "dataset", "track", "session"], how="inner")
    latest_runs = _ordered(latest_runs, ["session"])
    latest_buckets = _ordered(latest_buckets, ["bucket_min_count", "session"])

print(f"Discovered {len(all_runs)} cold-start runs across {sum(source['root'].exists() for source in ARTIFACT_SOURCES)} artifact roots.")


## Table 1: Latest Run Overview

This table keeps one latest session per `(model, dataset, track)` tuple and shows the overall cold-start metrics stored in the summary payload.


In [ ]:
if latest_runs.empty:
    print("No cold-start runs available.")
else:
    overview = latest_runs[
        [
            "model",
            "dataset",
            "track_label",
            "session",
            "bucket_spec",
            "recall@5",
            "ndcg@5",
            "recall@10",
            "ndcg@10",
            "checkpoint_name",
            "artifact_root",
        ]
    ].copy()
    overview = overview.rename(columns={"track_label": "track"})
    display(overview)


## Table 2: Bucket-Level Metrics

The first table pivots the primary metric by training-frequency bucket. The second table shows how many test examples fall into each bucket.


In [ ]:
if latest_buckets.empty:
    print("No bucket-level rows available.")
else:
    metric_table = latest_buckets.pivot_table(
        index=["model", "dataset", "track_label"],
        columns="bucket_range",
        values=PRIMARY_METRIC,
        aggfunc="first",
    )
    metric_table = _ordered(metric_table.reset_index(), [])
    metric_table = metric_table.rename(columns={"track_label": "track"})
    print(f"Primary metric by bucket: {PRIMARY_METRIC}")
    display(metric_table)

    count_table = latest_buckets.pivot_table(
        index=["model", "dataset", "track_label"],
        columns="bucket_range",
        values="n_examples",
        aggfunc="first",
    )
    count_table = _ordered(count_table.reset_index(), [])
    count_table = count_table.rename(columns={"track_label": "track"})
    print("Test examples per bucket")
    display(count_table)


## Plot 1: Primary Metric by Bucket

Each subplot corresponds to one dataset. Bars are grouped by reproduced model and config track.


In [ ]:
if latest_buckets.empty:
    print("No bucket-level rows available.")
else:
    datasets = [name for name in DATASET_ORDER if name in set(latest_buckets["dataset"])]
    if not datasets:
        datasets = sorted(set(latest_buckets["dataset"]))
    figure, axes = plt.subplots(len(datasets), 1, figsize=(12, 3.8 * len(datasets)), constrained_layout=True)
    if len(datasets) == 1:
        axes = [axes]

    color_map = {
        "RPG | released README": "#355C7D",
        "RPG | paper appendix": "#C06C84",
        "RPG | legacy": "#6C5B7B",
        "SASRec | released README": "#2A9D8F",
        "SASRec | paper appendix": "#E9C46A",
        "SASRec | legacy": "#A8DADC",
    }

    for axis, dataset in zip(axes, datasets):
        frame = latest_buckets[latest_buckets["dataset"] == dataset].copy()
        buckets = frame.sort_values("bucket_min_count")["bucket_range"].drop_duplicates().tolist()
        series = (
            frame[["model", "track_label"]]
            .drop_duplicates()
            .sort_values(["model", "track_label"], key=lambda col: col)
            .apply(lambda row: f"{row['model']} | {row['track_label']}", axis=1)
            .tolist()
        )
        series = [label for label in series if label.split(" | ")[0] in MODEL_ORDER]
        x = np.arange(len(buckets), dtype=float)
        width = 0.8 / max(len(series), 1)

        for index, label in enumerate(series):
            model, track = label.split(" | ", 1)
            series_frame = frame[(frame["model"] == model) & (frame["track_label"] == track)].sort_values("bucket_min_count")
            values = [
                float(series_frame.loc[series_frame["bucket_range"] == bucket, PRIMARY_METRIC].iloc[0])
                if not series_frame.loc[series_frame["bucket_range"] == bucket, PRIMARY_METRIC].empty
                else np.nan
                for bucket in buckets
            ]
            offsets = x - 0.4 + width / 2 + index * width
            axis.bar(offsets, values, width=width, label=label, color=color_map.get(label, None))

        axis.set_title(dataset)
        axis.set_xlabel("Training frequency bucket")
        axis.set_ylabel(PRIMARY_METRIC)
        axis.set_xticks(x)
        axis.set_xticklabels(buckets)
        axis.grid(axis="y", alpha=0.25)
        axis.set_ylim(bottom=0)
        axis.legend(loc="upper left")

    plt.show()


## Plot 2: Test Example Counts by Bucket

This plot helps interpret whether a bucket-level metric is based on a large or small slice of the test set.


In [ ]:
if latest_buckets.empty:
    print("No bucket-level rows available.")
else:
    datasets = [name for name in DATASET_ORDER if name in set(latest_buckets["dataset"])]
    if not datasets:
        datasets = sorted(set(latest_buckets["dataset"]))
    figure, axes = plt.subplots(len(datasets), 1, figsize=(12, 3.8 * len(datasets)), constrained_layout=True)
    if len(datasets) == 1:
        axes = [axes]

    color_map = {
        "RPG | released README": "#355C7D",
        "RPG | paper appendix": "#C06C84",
        "RPG | legacy": "#6C5B7B",
        "SASRec | released README": "#2A9D8F",
        "SASRec | paper appendix": "#E9C46A",
        "SASRec | legacy": "#A8DADC",
    }

    for axis, dataset in zip(axes, datasets):
        frame = latest_buckets[latest_buckets["dataset"] == dataset].copy()
        buckets = frame.sort_values("bucket_min_count")["bucket_range"].drop_duplicates().tolist()
        series = (
            frame[["model", "track_label"]]
            .drop_duplicates()
            .sort_values(["model", "track_label"], key=lambda col: col)
            .apply(lambda row: f"{row['model']} | {row['track_label']}", axis=1)
            .tolist()
        )
        x = np.arange(len(buckets), dtype=float)
        width = 0.8 / max(len(series), 1)

        for index, label in enumerate(series):
            model, track = label.split(" | ", 1)
            series_frame = frame[(frame["model"] == model) & (frame["track_label"] == track)].sort_values("bucket_min_count")
            values = [
                float(series_frame.loc[series_frame["bucket_range"] == bucket, "n_examples"].iloc[0])
                if not series_frame.loc[series_frame["bucket_range"] == bucket, "n_examples"].empty
                else np.nan
                for bucket in buckets
            ]
            offsets = x - 0.4 + width / 2 + index * width
            axis.bar(offsets, values, width=width, label=label, color=color_map.get(label, None))

        axis.set_title(dataset)
        axis.set_xlabel("Training frequency bucket")
        axis.set_ylabel("n_examples")
        axis.set_xticks(x)
        axis.set_xticklabels(buckets)
        axis.grid(axis="y", alpha=0.25)
        axis.legend(loc="upper right")

    plt.show()


## Paper Figure 5 Reproduction: Sports Cold Start

The paper's Figure 5 reports Sports `NDCG@10` across the four cold-start buckets. This cell reproduces that view for all available reproduced models.


In [ ]:
if latest_buckets.empty:
    print("No bucket-level rows available.")
else:
    sports = latest_buckets[latest_buckets["dataset"] == "Sports"].copy()
    if sports.empty:
        print("No Sports cold-start runs available.")
    else:
        buckets = sports.sort_values("bucket_min_count")["bucket_range"].drop_duplicates().tolist()
        series = (
            sports[["model", "track_label"]]
            .drop_duplicates()
            .sort_values(["model", "track_label"], key=lambda col: col)
            .apply(lambda row: f"{row['model']} | {row['track_label']}", axis=1)
            .tolist()
        )
        x = np.arange(len(buckets), dtype=float)
        width = 0.8 / max(len(series), 1)
        color_map = {
            "RPG | released README": "#355C7D",
            "RPG | paper appendix": "#C06C84",
            "RPG | legacy": "#6C5B7B",
            "SASRec | released README": "#2A9D8F",
            "SASRec | paper appendix": "#E9C46A",
            "SASRec | legacy": "#A8DADC",
        }

        fig, ax = plt.subplots(figsize=(9.5, 4.8), constrained_layout=True)
        for index, label in enumerate(series):
            model, track = label.split(" | ", 1)
            frame = sports[(sports["model"] == model) & (sports["track_label"] == track)].sort_values("bucket_min_count")
            values = [
                float(frame.loc[frame["bucket_range"] == bucket, PRIMARY_METRIC].iloc[0])
                if not frame.loc[frame["bucket_range"] == bucket, PRIMARY_METRIC].empty
                else np.nan
                for bucket in buckets
            ]
            offsets = x - 0.4 + width / 2 + index * width
            ax.bar(offsets, values, width=width, label=label, color=color_map.get(label, None))

        ax.set_title("Sports Cold-Start Analysis")
        ax.set_xlabel("Training frequency bucket")
        ax.set_ylabel(PRIMARY_METRIC)
        ax.set_xticks(x)
        ax.set_xticklabels(buckets)
        ax.grid(axis="y", alpha=0.25)
        ax.set_ylim(bottom=0)
        ax.legend(loc="upper left")
        plt.show()
